In [1]:
from gliclass.data_processing import GLiClassDataset
from transformers import AutoModel, AutoConfig
from gliclass.config import GLiClassModelConfig
from gliclass.model import GLiClassModel, GLiClassBiEncoder, GLiClassAudio
from transformers import AutoConfig, AutoTokenizer
from transformers import Wav2Vec2Model, Wav2Vec2FeatureExtractor
import torchaudio, torch

/home/werent4/GLiClass/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/werent4/GLiClass/.venv/lib/python3.10/site-packages/transformers/utils/hub.py:128: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


In [2]:
audiocfg = AutoConfig.from_pretrained("facebook/wav2vec2-base-960h")
deberta_cfg = AutoConfig.from_pretrained("microsoft/deberta-v3-small")
tokenizer = AutoTokenizer.from_pretrained("microsoft/deberta-v3-small")
audio_feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained(
    "facebook/wav2vec2-base-960h"
)


/home/werent4/GLiClass/.venv/lib/python3.10/site-packages/transformers/convert_slow_tokenizer.py:561: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


In [3]:
glicalss_config = GLiClassModelConfig(
    encoder_config=deberta_cfg,
    encoder_model="microsoft/deberta-v3-small",
    audio_model_name="facebook/wav2vec2-base-960h",
    audio_model_config=audiocfg,
    class_token_index=len(tokenizer),
    text_token_index=len(tokenizer)+1,
    audio_token_index=len(tokenizer)+2, 
    pooling_strategy="first",
    scorer_type="simple",
    use_lstm=False,
    focal_loss_alpha=-1,
    focal_loss_gamma=-1,
    contrastive_loss_coef=0.0,
    normalize_features=False,
    extract_text_features=False,
    architecture_type='audio-bi-encoder',
    prompt_first=True,
    squeeze_layers=False,
    shuffle_labels=True
)

model = GLiClassModel(glicalss_config, from_pretrained=False)
new_words = ["<<LABEL>>", "<<SEP>>", "<<AUDIO>>"]
tokenizer.add_tokens(new_words, special_tokens=True)
model.resize_token_embeddings(len(tokenizer), None)

Embedding(128004, 768, padding_idx=0)

In [4]:
import json
data = json.load(open("./datasets/processed_dataset.json", "r"))

In [5]:
train_dataset = GLiClassDataset(data, tokenizer, 1024, 
                                'multi_label_classification', "audio-bi-encoder", 
                                True, labels_tokenizer=tokenizer)
exmpl = train_dataset[0]

Total labels:  7


In [6]:
exmpl

{'input_ids': [[1, 9424, 66619, 569, 2], [1, 32168, 2, 0, 0], [1, 3916, 2, 0, 0], [1, 15653, 2, 0, 0], [1, 29504, 14341, 407, 2], [1, 13142, 6149, 2, 0], [1, 27927, 2, 0, 0]], 'token_type_ids': [[0, 0, 0, 0, 0], [0, 0, 0, 0, 0], [0, 0, 0, 0, 0], [0, 0, 0, 0, 0], [0, 0, 0, 0, 0], [0, 0, 0, 0, 0], [0, 0, 0, 0, 0]], 'attention_mask': [[1, 1, 1, 1, 1], [1, 1, 1, 0, 0], [1, 1, 1, 0, 0], [1, 1, 1, 0, 0], [1, 1, 1, 1, 1], [1, 1, 1, 1, 0], [1, 1, 1, 0, 0]], 'labels_mask': tensor([1., 1., 1., 1., 1., 1., 1.]), 'labels': tensor([1., 0., 0., 0., 0., 0., 0.]), 'labels_text': ['Disgusted', 'Neutral', 'Happy', 'Sad', 'Suprised', 'Fearful', 'Angry'], 'audio_input': tensor([0.0208, 0.0175, 0.0168,  ..., 0.0004, 0.0004, 0.0004])}

In [7]:
tokenizer.decode(exmpl['input_ids'][3])

'[CLS] Sad[SEP][PAD][PAD]'

In [8]:
print("input_ids type:", type(exmpl['input_ids']))
print("attention_mask type:", type(exmpl['attention_mask']))
print("labels type:", type(exmpl['labels']))

input_ids type: <class 'list'>
attention_mask type: <class 'list'>
labels type: <class 'torch.Tensor'>


In [9]:
input_ids = torch.tensor(exmpl['input_ids']).unsqueeze(0)  
attention_mask = torch.tensor(exmpl['attention_mask']).unsqueeze(0)  
labels = torch.tensor(exmpl['labels']).unsqueeze(0)  

/var/tmp/ipykernel_29360/2331616996.py:3: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  labels = torch.tensor(exmpl['labels']).unsqueeze(0)


In [10]:
exmpl['labels']

tensor([1., 0., 0., 0., 0., 0., 0.])

In [11]:
print("Labels shape:", labels.shape)
print("Labels:", labels)
print("Problem type:", model.config.problem_type)

Labels shape: torch.Size([1, 7])
Labels: tensor([[1., 0., 0., 0., 0., 0., 0.]])
Problem type: None


In [12]:
exmpl['audio_input'].unsqueeze(0).shape

torch.Size([1, 80000])

In [14]:
model(input_ids, attention_mask, exmpl["audio_input"], labels=labels, return_dict=True)

GLiClassOutput(loss=tensor(4.1938, grad_fn=<NegBackward0>), logits=tensor([[-0.5545,  3.1775,  0.8603,  0.8491,  1.9720, -0.0429, -0.4186]],
       grad_fn=<MulBackward0>), hidden_states=None, attentions=None, text_embeddings=None, class_embeddings=None)